In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

## I. Read Historical Prices Data

In [0]:
jdbc_url = "jdbc:sqlserver://capstone-database-server.database.windows.net:1433;database=writedatabasesilverlayer;"
connection_properties = {
    "user": "capstonedioxieteam",
    "password": "Connhenbeo1@",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

historical_prices_dataframe = spark.read.jdbc(
    url=jdbc_url,
    table="Silver.Historical_Prices",
    properties=connection_properties
)

## II. Read Historical Prices TA Data

In [0]:
historical_prices_ta_dataframe = spark.read.jdbc(
    url=jdbc_url,
    table="Silver.Historical_Prices_TA",
    properties=connection_properties
)

## III. Company Information Data

In [0]:
company_information_dataframe = spark.read.jdbc(
    url=jdbc_url,
    table="Silver.Company_Information",
    properties=connection_properties
)

In [0]:
display(historical_prices_dataframe)

In [0]:
historical_prices_dataframe_date_process = historical_prices_dataframe   \
    .withColumn("Date", to_date(substring(col("Date"), 1, 10)))

In [0]:
df_prices_with_ta = historical_prices_dataframe_date_process    \
    .join(historical_prices_ta_dataframe, ["Stock_Symbol", "Date", "Close"], "inner")

company_information_dataframe = company_information_dataframe   \
    .withColumnRenamed("symbol", "Stock_Symbol")

df_prices_with_ta_with_company_information = df_prices_with_ta  \
    .join(company_information_dataframe, ["Stock_Symbol"], "left")

In [0]:
# list_schema = df_prices_with_ta_with_company_information.schema

# sql_string_create_table_head = """
# IF NOT EXISTS (
#     SELECT * FROM INFORMATION_SCHEMA.TABLES 
#     WHERE TABLE_NAME = 'Historical_Prices_Stock_with_TA_Company_Information' AND TABLE_SCHEMA = 'Gold'
# )
# BEGIN
#     CREATE TABLE Gold.Historical_Prices_Stock_with_TA_Company_Information (
# """

# sql_string_create_table_tail = """
#     );
# END
# """

# sql_columns_and_data_types = ""

# for schema_tuple in list_schema:
#     start_index = str(schema_tuple.typeName).find(",", 0)
#     end_index = str(schema_tuple.typeName).find(",", start_index + 1)
#     data_type_schema=str(schema_tuple.typeName)[start_index + 2:end_index][:-6]
#     if data_type_schema == "String":
#         max_length = df_prices_with_ta_with_company_information.select(max(length(col(schema_tuple.name)))).collect()[0][0]
#         data_type_schema = f"NVARCHAR({max_length+1})"
#     elif data_type_schema == "Timestamp":
#         data_type_schema = "DATETIME"
#     elif data_type_schema == "Integer":
#         data_type_schema = "INT"
#     elif data_type_schema == "Double":
#         data_type_schema = "FLOAT"
#     elif data_type_schema == "Boolean":
#         data_type_schema = "BIT"
#     elif data_type_schema == "Date":
#         data_type_schema = "DATE"
#     elif data_type_schema == "Long":
#         data_type_schema = "BIGINT"
#     else:
#         print(f"Data type {data_type_schema} not supported")
    
#     sql_columns_and_data_types += f"\n\t[{schema_tuple.name}] {data_type_schema} NULL,"

# full_sql_create_table = sql_string_create_table_head + \
#                         sql_columns_and_data_types +  \
#                         sql_string_create_table_tail
# print(full_sql_create_table)

In [0]:
df_prices_with_ta_with_company_information.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "Gold.Historical_Prices_Stock_with_TA_Company_Information") \
    .option("user", connection_properties["user"]) \
    .option("password", connection_properties["password"]) \
    .option("driver", connection_properties["driver"]) \
    .mode("overwrite") \
    .option("batchsize", 10000) \
    .option("numPartitions", 8) \
    .save()

print("Data successfully written to Azure SQL Database.")